In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'usecase':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))


# Use Case 3: Rates Desk Smile-Aware Quoting

## Business Scenario

A rates trading desk needs to quote and mark a **2Y x 5Y payer swaption smile slice**.
Using a single flat ATM volatility is often not enough, because clients may ask for strikes away from ATM.
A practical desk workflow is therefore:

1. Observe a market smile slice for one expiry-tenor point
2. Calibrate SABR parameters to the slice
3. Use the fitted smile to produce strike-consistent implied vols and prices
4. Compare fitted outputs against the raw market quotes used in the desk mark


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from swaption_pricing.black76 import price_swaption
from swaption_pricing.calibration import calibrate_sabr_to_vols, calibration_diagnostics, calibration_rows
from swaption_pricing.data_loader import load_curve_points_csv, load_swaption_vol_slice_csv
from swaption_pricing.market_validation import load_json_metadata
from swaption_pricing.sabr import price_swaption_with_sabr
from swaption_pricing.swap import forward_swap_rate, swap_annuity
from swaption_pricing.types import SwaptionSpec


## Market Inputs

This use case combines two inputs:

- A public U.S. Treasury yield-curve proxy for discounting and forward-rate generation
- A **market-like 2Y x 5Y Black-vol smile slice** stored in the project as a representative quoting input

The curve is public-data-based; the smile slice is illustrative and designed to resemble a plausible desk quoting surface.


In [ ]:
curve_path = PROJECT_ROOT / 'data/raw/market/ust_yield_curve_proxy/curve_points.csv'
metadata_path = PROJECT_ROOT / 'data/raw/market/ust_yield_curve_proxy/ust_yield_curve_snapshot.json'
vol_slice_path = PROJECT_ROOT / 'data/raw/example/vol_slice.csv'

curve = load_curve_points_csv(curve_path)
curve_metadata = load_json_metadata(metadata_path)
vol_quotes = load_swaption_vol_slice_csv(vol_slice_path)

curve_metadata


## Desk Slice Specification

Representative desk quoting setup:

- Notional used for comparison: `USD 100,000,000`
- Expiry: `2Y`
- Underlying swap tenor: `5Y`
- Option type: `payer`
- Strike grid: market-like smile slice in `vol_slice.csv`


In [ ]:
notional = 100_000_000.0
expiry = vol_quotes[0].expiry
tenor = vol_quotes[0].tenor
pay_frequency = 1
option_type = 'payer'

strikes = [quote.strike for quote in vol_quotes]
market_vols = [quote.vol for quote in vol_quotes]
forward = forward_swap_rate(curve, expiry, tenor, pay_frequency)
annuity = swap_annuity(curve, expiry, tenor, pay_frequency)

desk_slice_summary = pd.DataFrame([
    {
        'curve_snapshot_date': curve_metadata['yield_curve_date'],
        'expiry': expiry,
        'tenor': tenor,
        'forward_swap_rate': forward,
        'annuity': annuity,
        'notional': notional,
    }
])
desk_slice_summary


## Observed Market-Like Smile Slice

These quotes represent the desk's raw strike-vol inputs for this expiry-tenor point.
In real markets, these would usually come from broker runs, vendor screens, or internal marking sources.


In [ ]:
market_slice_df = pd.DataFrame([
    {
        'strike': quote.strike,
        'market_vol': quote.vol,
        'moneyness_vs_forward_bp': (quote.strike - forward) * 10000,
    }
    for quote in vol_quotes
])
market_slice_df


## SABR Calibration

The desk calibrates SABR to convert the raw smile slice into a smooth, strike-consistent volatility function.
Here we fix `beta = 0.5` and solve for `alpha`, `rho`, and `nu`.


In [ ]:
calibration_result = calibrate_sabr_to_vols(
    forward=forward,
    strikes=strikes,
    expiry=expiry,
    market_vols=market_vols,
    beta=0.5,
)
diagnostics = calibration_diagnostics(calibration_result)

calibration_summary = pd.DataFrame([
    {
        'alpha': calibration_result.params.alpha,
        'beta': calibration_result.params.beta,
        'rho': calibration_result.params.rho,
        'nu': calibration_result.params.nu,
        'objective_value': calibration_result.objective_value,
        'rmse': diagnostics.rmse,
        'max_abs_error': diagnostics.max_abs_error,
    }
])
calibration_summary


In [ ]:
fit_df = pd.DataFrame([
    {
        'strike': row.strike,
        'market_vol': row.market_vol,
        'fitted_vol': row.fitted_vol,
        'residual': row.residual,
    }
    for row in calibration_rows(calibration_result)
])
fit_df


## Strike-by-Strike Pricing Comparison

The desk can now compare two pricing approaches:

- `Black using observed market vol at each strike`
- `Black using SABR-fitted implied vol at each strike`

If calibration is good, the two should be close, while the SABR version gives a consistent smile function rather than isolated points.


In [ ]:
pricing_rows = []
for quote in vol_quotes:
    spec = SwaptionSpec(
        notional=notional,
        expiry=expiry,
        tenor=tenor,
        strike=quote.strike,
        pay_frequency=pay_frequency,
        option_type=option_type,
    )
    black_price = price_swaption(curve, spec, quote.vol)
    sabr_price, sabr_implied_vol = price_swaption_with_sabr(curve, spec, calibration_result.params)
    pricing_rows.append({
        'strike': quote.strike,
        'market_vol': quote.vol,
        'sabr_implied_vol': sabr_implied_vol,
        'black_market_quote_price': black_price,
        'sabr_fitted_price': sabr_price,
        'price_difference': sabr_price - black_price,
    })

pricing_df = pd.DataFrame(pricing_rows)
pricing_df


In [ ]:
ax = fit_df.plot(x='strike', y=['market_vol', 'fitted_vol'], marker='o', figsize=(8, 4), title='Market vs SABR-Fitted Volatility Slice')
ax.set_ylabel('Black Volatility')
plt.show()

ax = fit_df.plot(x='strike', y='residual', kind='bar', figsize=(8, 4), title='Calibration Residual by Strike')
ax.set_ylabel('Fitted - Market Vol')
plt.xticks(rotation=0)
plt.show()

ax = pricing_df.plot(x='strike', y=['black_market_quote_price', 'sabr_fitted_price'], marker='o', figsize=(8, 4), title='Black vs SABR Price by Strike')
ax.set_ylabel('Price')
plt.show()


## Desk Interpretation

- A desk rarely wants to quote each strike independently with no smile structure behind it.
- SABR provides a smooth interpolation layer that is consistent with the observed strike slice.
- The fitted vols in this example are very close to the observed quotes, with a small calibration RMSE.
- The pricing comparison shows that SABR can reproduce the desk's market-like quotes while also supporting consistent marking for strikes that may not trade directly.
- In a real desk environment, the market smile would come from broker runs or vendor feeds rather than an internal illustrative CSV, but the workflow shown here is the same.
